# CARec encoder experiments

- Run shared setup and function cells once, then run only the model sections you choose.
- Each variant has one fixed folder containing only its two best model checkpoints. Rerunning replaces them.
- Metrics stay in notebook outputs and memory. Automatic resume is not implemented.


## Shared settings

- Model and MLP run on MPS; sparse graph propagation and uniformity run on CPU.
- Settings below apply to every model. Set both epoch limits to 1 for a short check.


In [23]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "requirements.txt").is_file()
)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from helper import Experiment, RESULT_FILES

DATASET_ID = "electronics_carec_035_seed2023"
seed = 2023
phase1_epochs = 90
phase2_epochs = 30
patience = 15
weight_decay = 0.0
training_batch_size = 2048
learning_rate = 0.001
device = "mps"


## Shared data

- Load interactions and RecBole ID mappings once. Keep the saved splits and repeated interactions.
- Padding has ID 0. Only training edges enter the model graph.


In [ ]:
experiment = Experiment(
    PROJECT_ROOT, DATASET_ID, device=device, seed=seed,
    phase1_epochs=phase1_epochs, phase2_epochs=phase2_epochs, patience=patience,
    weight_decay=weight_decay, training_batch_size=training_batch_size,
    learning_rate=learning_rate,
)


## Content-only baseline

- Each model cell evaluates CARec, then content-only through `helper.py`.
- The baseline averages cached vectors over training interactions and ranks by cosine similarity on CPU, with the same evaluation masks and metrics.
- Results are saved separately in JSON. The final table reads both methods from disk.


## Individual runs

- Run each cell independently after the shared cells above. Definitions alone do not start training.
- Returned values contain metrics and paths, not the trained model.


### MiniLM-L6-v2

- Input: without task instruction. The exact prompt is read from the cache and saved with the run.


In [ ]:
minilm_result = experiment.run_experiment("all-MiniLM-L6-v2.npz")
minilm_content_result = experiment.evaluate_content_only("all-MiniLM-L6-v2.npz")


/var/folders/wz/x7xqj0yj6ls_367syhygxjn40000gn/T/ipykernel_65941/822926853.py:45: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:55.)
  ).coalesce().to_sparse_csr()


Evaluation only: all-MiniLM-L6-v2.npz, loaded phase2-best.pt


Cold: 100%|██████████| 292/292 [00:02<00:00, 117.75it/s]


CARec


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0714,0.1050,0.0619,0.0702
cold,0.0532,0.1344,0.0304,0.0488


Content-only cold: 100%|██████████| 292/292 [00:01<00:00, 174.24it/s]


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0269,0.0489,0.0194,0.0247
cold,0.0479,0.1222,0.0290,0.0454


Zero-profile users: {'warm': 0, 'cold': 0}


### MPNet-base-v2

- Input: without task instruction. The exact prompt is read from the cache and saved with the run.


In [ ]:
mpnet_result = experiment.run_experiment("all-mpnet-base-v2.npz")
mpnet_content_result = experiment.evaluate_content_only("all-mpnet-base-v2.npz")


Evaluation only: all-mpnet-base-v2.npz, loaded phase2-best.pt


Cold: 100%|██████████| 292/292 [00:02<00:00, 99.27it/s] 

CARec


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0718,0.1046,0.0626,0.0708
cold,0.0551,0.1427,0.0326,0.0524


Content-only cold: 100%|██████████| 292/292 [00:01<00:00, 147.52it/s]


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0258,0.0456,0.0183,0.0232
cold,0.0502,0.1253,0.0300,0.0470


Zero-profile users: {'warm': 0, 'cold': 0}


### E5-base-v2

- Input: without task instruction, query prefix. The exact prompt is read from the cache and saved with the run.


In [ ]:
e5_base_result = experiment.run_experiment("e5-base-v2.npz")
e5_base_content_result = experiment.evaluate_content_only("e5-base-v2.npz")


Evaluation only: e5-base-v2.npz, loaded phase2-best.pt


Cold: 100%|██████████| 292/292 [00:02<00:00, 105.81it/s]

CARec


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0721,0.1051,0.0630,0.0711
cold,0.0530,0.1364,0.0305,0.0493


Content-only cold: 100%|██████████| 292/292 [00:01<00:00, 147.70it/s]


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0359,0.0576,0.0275,0.0328
cold,0.0532,0.1312,0.0332,0.0506


Zero-profile users: {'warm': 0, 'cold': 0}


### Qwen3-Embedding-0.6B

- Input: without task instruction. The exact prompt is read from the cache and saved with the run.


In [ ]:
qwen_plain_result = experiment.run_experiment("Qwen3-Embedding-0.6B-without_instruction.npz", phase2_only=True)
qwen_plain_content_result = experiment.evaluate_content_only("Qwen3-Embedding-0.6B-without_instruction.npz")


Evaluation only: Qwen3-Embedding-0.6B-without_instruction.npz, loaded phase2-best.pt


Cold: 100%|██████████| 292/292 [00:03<00:00, 91.89it/s] 

CARec


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0724,0.1069,0.0627,0.0712
cold,0.0588,0.1431,0.0352,0.0542


Content-only cold: 100%|██████████| 292/292 [00:02<00:00, 129.97it/s]


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0199,0.0373,0.0142,0.0184
cold,0.0427,0.1015,0.0259,0.0391


Zero-profile users: {'warm': 0, 'cold': 0}


### Qwen3-Embedding-0.6B

- Input: with task instruction. The exact prompt is read from the cache and saved with the run.


In [ ]:
qwen_instruct_result = experiment.run_experiment("Qwen3-Embedding-0.6B-with_instruction.npz")
qwen_instruct_content_result = experiment.evaluate_content_only("Qwen3-Embedding-0.6B-with_instruction.npz")


Evaluation only: Qwen3-Embedding-0.6B-with_instruction.npz, loaded phase2-best.pt


Cold: 100%|██████████| 292/292 [00:02<00:00, 97.59it/s] 


CARec


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0724,0.1050,0.063,0.0711
cold,0.0534,0.1367,0.031,0.0497


Content-only cold: 100%|██████████| 292/292 [00:02<00:00, 131.02it/s]


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0199,0.0382,0.0143,0.0187
cold,0.0455,0.1158,0.0264,0.0421


Zero-profile users: {'warm': 0, 'cold': 0}


### Multilingual E5-large-instruct

- Input: without task instruction. The exact prompt is read from the cache and saved with the run.


In [ ]:
e5_large_plain_result = experiment.run_experiment("multilingual-e5-large-instruct-without_instruction.npz")
e5_large_plain_content_result = experiment.evaluate_content_only("multilingual-e5-large-instruct-without_instruction.npz")


Evaluation only: multilingual-e5-large-instruct-without_instruction.npz, loaded phase2-best.pt


Cold: 100%|██████████| 292/292 [00:02<00:00, 101.20it/s]


CARec


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0723,0.1053,0.0631,0.0713
cold,0.0555,0.1405,0.0321,0.0514


Content-only cold: 100%|██████████| 292/292 [00:02<00:00, 120.43it/s]


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0325,0.0543,0.0247,0.0300
cold,0.0530,0.1267,0.0329,0.0495


Zero-profile users: {'warm': 0, 'cold': 0}


### Multilingual E5-large-instruct

- Input: with task instruction. The exact prompt is read from the cache and saved with the run.


In [ ]:
e5_large_instruct_result = experiment.run_experiment("multilingual-e5-large-instruct-with_instruction.npz",phase2_only=True)
e5_large_instruct_content_result = experiment.evaluate_content_only("multilingual-e5-large-instruct-with_instruction.npz")


Evaluation only: multilingual-e5-large-instruct-with_instruction.npz, loaded phase2-best.pt


Cold: 100%|██████████| 292/292 [00:02<00:00, 97.60it/s] 

CARec


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0719,0.1050,0.063,0.0712
cold,0.0506,0.1353,0.029,0.0482


Content-only cold: 100%|██████████| 292/292 [00:02<00:00, 136.28it/s]


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0267,0.0466,0.0203,0.0251
cold,0.0485,0.1202,0.0294,0.0455


Zero-profile users: {'warm': 0, 'cold': 0}


### INSTRUCTOR-base

- Input: without task instruction. The exact prompt is read from the cache and saved with the run.


In [ ]:
instructor_plain_result = experiment.run_experiment("instructor-base-without_instruction.npz")
instructor_plain_content_result = experiment.evaluate_content_only("instructor-base-without_instruction.npz")



Evaluation only: instructor-base-without_instruction.npz, loaded phase2-best.pt


Cold: 100%|██████████| 292/292 [00:02<00:00, 107.12it/s]

CARec


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0719,0.1055,0.0628,0.0711
cold,0.0516,0.1372,0.0287,0.0479


Content-only cold: 100%|██████████| 292/292 [00:02<00:00, 145.06it/s]


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0239,0.0430,0.0172,0.0218
cold,0.0441,0.1128,0.0258,0.0410


Zero-profile users: {'warm': 0, 'cold': 0}


### INSTRUCTOR-base

- Input: with task instruction. The exact prompt is read from the cache and saved with the run.


In [ ]:
instructor_instruct_result = experiment.run_experiment("instructor-base-with_instruction.npz")
instructor_instruct_content_result = experiment.evaluate_content_only("instructor-base-with_instruction.npz")


Evaluation only: instructor-base-with_instruction.npz, loaded phase2-best.pt


Cold: 100%|██████████| 292/292 [00:02<00:00, 105.02it/s]

CARec


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0721,0.1048,0.0631,0.0712
cold,0.0470,0.1316,0.0265,0.0455


Content-only cold: 100%|██████████| 292/292 [00:02<00:00, 144.05it/s]


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0229,0.0422,0.0164,0.0211
cold,0.0432,0.1134,0.0256,0.0413


Zero-profile users: {'warm': 0, 'cold': 0}


## Results

- Show CARec and content-only metrics from this session and saved JSON files. Each method can appear independently.
- Save the table as `results/carec/<dataset>/summary.csv`. Show variant, method, dimension, and warm/cold metrics only.
- After a kernel restart, run the experiment setup and this cell to display saved results without evaluation or training.


In [ ]:
saved_results = {method: experiment.load_experiment_results(method) for method in RESULT_FILES}
variants = sorted({variant for results in saved_results.values() for variant in results})
result_rows = []
for variant in variants:
    for method, results in saved_results.items():
        if variant not in results:
            continue
        result = results[variant]
        row = {"variant": variant, "method": method, "dimension": result["dimension"]}
        for split_name, metrics in result["test"].items():
            row.update({f"{split_name}_{metric}": value for metric, value in metrics.items()})
        result_rows.append(row)
summary = pd.DataFrame(result_rows)
if not summary.empty:
    experiment.results_dir.mkdir(parents=True, exist_ok=True)
    temporary = experiment.results_dir / "summary.csv.tmp"
    summary.to_csv(temporary, index=False)
    temporary.replace(experiment.results_dir / "summary.csv")
display(summary)


,variant,method,dimension,warm_recall@10,warm_recall@50,warm_ndcg@10,warm_ndcg@50,cold_recall@10,cold_recall@50,cold_ndcg@10,cold_ndcg@50
0,Qwen3-Embedding-0.6B-with_instruction,CARec,1024,0.0724,0.1050,0.0630,0.0711,0.0534,0.1367,0.0310,0.0497
1,Qwen3-Embedding-0.6B-with_instruction,Content-only,1024,0.0199,0.0382,0.0143,0.0187,0.0455,0.1158,0.0264,0.0421
2,Qwen3-Embedding-0.6B-without_instruction,CARec,1024,0.0724,0.1069,0.0627,0.0712,0.0588,0.1431,0.0352,0.0542
3,Qwen3-Embedding-0.6B-without_instruction,Content-only,1024,0.0199,0.0373,0.0142,0.0184,0.0427,0.1015,0.0259,0.0391
4,all-MiniLM-L6-v2,CARec,384,0.0714,0.1050,0.0619,0.0702,0.0532,0.1344,0.0304,0.0488
5,all-MiniLM-L6-v2,Content-only,384,0.0269,0.0489,0.0194,0.0247,0.0479,0.1222,0.0290,0.0454
6,all-mpnet-base-v2,CARec,768,0.0718,0.1046,0.0626,0.0708,0.0551,0.1427,0.0326,0.0524
7,all-mpnet-base-v2,Content-only,768,0.0258,0.0456,0.0183,0.0232,0.0502,0.1253,0.0300,0.0470
8,e5-base-v2,CARec,768,0.0721,0.1051,0.0630,0.0711,0.0530,0.1364,0.0305,0.0493
9,e5-base-v2,Content-only,768,0.0359,0.0576,0.0275,0.0328,0.0532,0.1312,0.0332,0.0506
